In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# Установка seed для воспроизводимости результатов
keras.utils.set_random_seed(42)

I0000 00:00:1793450156.392249   35067 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793450158.115189   35067 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793450161.050512   35067 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# Загрузка данных (убедитесь, что файл heart.csv лежит в той же папке)
df = pd.read_csv("heart.csv")

# Проверка баланса классов (как на скриншоте)
print("Баланс классов:")
print(df.target.value_counts(normalize=True, dropna=False))

Баланс классов:
target
0    0.726073
1    0.273927
Name: proportion, dtype: float64


In [3]:
# Списки признаков
categorical_variables = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'ca', 'thal']
numerics = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'slope']

# One-Hot кодирование категориальных переменных
df1 = pd.get_dummies(df, columns=categorical_variables, dtype=int)
print(f"Размер датафрейма после кодирования: {df1.shape}")

Размер датафрейма после кодирования: (303, 30)


In [4]:
# Разделение на обучающую (80%) и тестовую (20%) выборки
test_df = df1.sample(frac=0.2, random_state=42)
train_df = df1.drop(test_df.index)

# Вычисление средних и стандартных отклонений ТОЛЬКО на train выборке
means = train_df[numerics].mean()
sd = train_df[numerics].std()

# Стандартизация (нормализация) числовых признаков
train_df[numerics] = (train_df[numerics] - means) / sd
test_df[numerics] = (test_df[numerics] - means) / sd

In [5]:
# Преобразование датафреймов в массивы NumPy
train = train_df.to_numpy()
test = test_df.to_numpy()

# Удаление столбца target (индекс 6) для получения признаков X
train_X = np.delete(train, 6, axis=1)
test_X = np.delete(test, 6, axis=1)

# Выделение столбца target (индекс 6) для получения целевой переменной y
train_y = train[:, 6]
test_y = test[:, 6]

print(f"Размер train_X: {train_X.shape}, размер test_X: {test_X.shape}")
print(f"Размер train_y: {train_y.shape}, размер test_y: {test_y.shape}")

Размер train_X: (242, 29), размер test_X: (61, 29)
Размер train_y: (242,), размер test_y: (61,)


In [6]:
num_columns = train_X.shape[1]

# Входной слой
input = keras.Input(shape=(num_columns,))

# Скрытый слой (16 нейронов, функция активации ReLU)
h = keras.layers.Dense(16, activation="relu", name="Hidden")(input)

# Выходной слой (1 нейрон, сигмоида для бинарной классификации)
output = keras.layers.Dense(1, activation="sigmoid", name="Output")(h)

# Сборка модели
model = keras.Model(input, output)
model.summary()

E0000 00:00:1793450246.567051   35067 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 29)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden (Dense)                  │ (None, 16)             │           480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 497 (1.94 KB)

 Trainable params: 497 (1.94 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
# Компиляция модели
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

# Обучение модели
history = model.fit(train_X, 
                    train_y, 
                    epochs=300,
                    batch_size=32,
                    verbose=1,
                    validation_split=0.2)

Epoch 1/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.5596 - loss: 0.7154 - val_accuracy: 0.5918 - val_loss: 0.6891
Epoch 2/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6062 - loss: 0.6826 - val_accuracy: 0.6122 - val_loss: 0.6697
Epoch 3/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6321 - loss: 0.6586 - val_accuracy: 0.6122 - val_loss: 0.6530
Epoch 4/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6528 - loss: 0.6376 - val_accuracy: 0.6122 - val_loss: 0.6372
Epoch 5/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6736 - loss: 0.6181 - val_accuracy: 0.6531 - val_loss: 0.6225
Epoch 6/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6995 - loss: 0.5995 - val_accuracy: 0.6939 - val_loss: 0.6088
Epoch 7/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7358 - loss: 0.5822 - val_accuracy: 0.6939 - val_loss: 0.5959
Epoch 8/300
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7513 - loss: 0.5658 - val_accuracy: 0.7143 - val_loss: 0.58

In [8]:
# Оценка качества на тестовых данных
test_loss, test_acc = model.evaluate(test_X, test_y)
print(f"\nТочность на тестовой выборке: {test_acc:.4f}")
print(f"Потери на тестовой выборке: {test_loss:.4f}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8361 - loss: 0.4810

Точность на тестовой выборке: 0.8361
Потери на тестовой выборке: 0.4810
